# 1. Conda 环境与首次构建 COF（Windows 版）

语言：[English](../en/01_environment_and_first_build.ipynb) | **中文**

本原生 Windows 版 Notebook 将创建仓库定义的 `cofkit` Conda 环境、把它注册为 Jupyter 内核，并在 `hcb` 拓扑上构建 TAPB–TFB 亚胺 COF。它使用 Windows PowerShell，不需要 WSL。请在本仓库的克隆目录中运行。

可执行单元格通过 Shell 命令调用 CLI。每个主要操作下方都提供了完全注释掉的 Python 单元格，用于展示等价的 API 工作流，但不会执行。

## 教程系列

1. **环境与首次构建**（本 Notebook）
2. [拓扑、连接数组合与连接键示例](02_topologies_connectivities_and_linkages.ipynb)
3. [使用 Zeo++ 进行孔隙分析](03_zeopp_pore_analysis.ipynb)

请从仓库根目录启动 Jupyter，确保下文所有相对路径都能一致解析。PowerShell 单元格使用 `conda run -n cofkit`，因此无需让内核的创建或激活状态在不同单元格之间保持。

## 查看仓库中的环境定义

`environment.yml` 的第一行将环境名称固定为 `cofkit`。整个教程系列都使用这个名称。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
Write-Host "Repository root: $root"
Write-Host ""
Get-Content -LiteralPath environment.yml | Select-Object -First 24

In [ ]:
# Python 等价实现（已注释）：查看环境名称。
# from pathlib import Path
# environment_name = next(
#     line.split(":", 1)[1].strip()
#     for line in Path("environment.yml").read_text().splitlines()
#     if line.startswith("name:")
# )
# print(environment_name)

## 创建或更新环境

这个单元格会先检查 `cofkit` CLI 是否已在 `PATH` 中可用。如果可用，单元格将成功退出且不修改环境，这在选择 **Run All**（全部运行）时尤其有用。否则，若 `cofkit` 环境不存在则创建它，存在则根据 `environment.yml` 更新；随后安装本地软件包而不替换 Conda 已解析的依赖，并为 Notebooks 2 和 3 注册 `Python (cofkit)` 内核。

> 首次运行时，环境依赖解析和 Open Babel 安装可能需要几分钟。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$cofkitCommand = Get-Command cofkit -ErrorAction SilentlyContinue
if ($null -ne $cofkitCommand) {
  Write-Host "cofkit CLI is already available at $($cofkitCommand.Source); skipping environment and package setup."
  exit 0
}
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
if ($null -eq (Get-Command conda -ErrorAction SilentlyContinue)) {
  throw "Conda is required. Install Miniforge or Miniconda, open its PowerShell prompt, and rerun this cell."
}
$environmentInfo = (& conda env list --json | ConvertFrom-Json)
if ($LASTEXITCODE -ne 0) { throw "conda env list failed with exit code $LASTEXITCODE." }
$environmentExists = @($environmentInfo.envs | Where-Object { (Split-Path -Leaf $_) -ieq "cofkit" }).Count -gt 0
if ($environmentExists) {
  & conda env update --name cofkit --file environment.yml --prune
} else {
  & conda env create --file environment.yml
}
if ($LASTEXITCODE -ne 0) { throw "Conda environment setup failed with exit code $LASTEXITCODE." }
& conda run -n cofkit python -m pip install --no-deps .
if ($LASTEXITCODE -ne 0) { throw "Local cofkit installation failed with exit code $LASTEXITCODE." }
& conda install --name cofkit --channel conda-forge --yes ipykernel
if ($LASTEXITCODE -ne 0) { throw "ipykernel installation failed with exit code $LASTEXITCODE." }
& conda run -n cofkit python -m ipykernel install --user --name cofkit --display-name "Python (cofkit)"
if ($LASTEXITCODE -ne 0) { throw "Jupyter kernel registration failed with exit code $LASTEXITCODE." }

In [ ]:
# 创建 Conda 环境没有对应的 Python API 实现。
# Conda 和 pip 是环境管理工具，因此应以上面的 Shell 命令为准。

如果本 Notebook 是用其他内核打开的，现在可以从 Jupyter 的内核菜单中选择 **Python (cofkit)**。后续可执行单元格都会显式使用 `conda run -n cofkit`，因此在这里切换内核并非必需。Notebooks 2 和 3 已将 `cofkit` 声明为默认内核。

## 验证安装

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
& conda run -n cofkit cofkit --version
if ($LASTEXITCODE -ne 0) { throw "cofkit version check failed with exit code $LASTEXITCODE." }
& conda run -n cofkit python -c "import cofkit; print('Python import:', cofkit.__version__)"
if ($LASTEXITCODE -ne 0) { throw "Python import check failed with exit code $LASTEXITCODE." }
& conda run -n cofkit cofkit build list-templates
if ($LASTEXITCODE -ne 0) { throw "Template listing failed with exit code $LASTEXITCODE." }

In [ ]:
# Python 等价实现（已注释）：验证软件包能够导入。
# import cofkit
# print(cofkit.__version__)
#
# from cofkit import ReactionLibrary
# library = ReactionLibrary.builtin()
# print(library)

## 在 `hcb` 拓扑上构建 TAPB–TFB

TAPB 和 TFB 各自具有三个反应基团，因此形成 3+3 连接数组合。`imine_bridge` 模板连接胺基和醛基，而 `--topology hcb` 则让拓扑选择明确且可复现。

该命令会写入 `summary.json`，并把导出的 CIF 按粗略验证类别存放在 `out/tutorial_01_first_build/cifs/` 下。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cofkitArguments = @(
  "build", "single-pair",
  "--template-id", "imine_bridge",
  "--first-smiles", "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
  "--second-smiles", "C1=C(C=C(C=C1C=O)C=O)C=O",
  "--first-id", "tapb",
  "--second-id", "tfb",
  "--first-motif-kind", "amine",
  "--second-motif-kind", "aldehyde",
  "--topology", "hcb",
  "--target-dimensionality", "2D",
  "--output-dir", "out/tutorial_01_first_build"
)
& conda run -n cofkit cofkit @cofkitArguments
if ($LASTEXITCODE -ne 0) { throw "COF build failed with exit code $LASTEXITCODE." }

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb",
#     "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine",
#     num_conformers=4,
# )
# tfb = build_rdkit_monomer(
#     "tfb",
#     "TFB",
#     "C1=C(C=C(C=C1C=O)C=O)C=O",
#     "aldehyde",
#     num_conformers=4,
# )
# generator = BatchStructureGenerator(
#     BatchGenerationConfig(
#         allowed_reactions=("imine_bridge",),
#         target_dimensionality="2D",
#         topology_ids=("hcb",),
#         rdkit_num_conformers=4,
#         max_workers=1,
#         write_cif=True,
#     )
# )
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb,
#     tfb,
#     pair_id="tapb__tfb",
#     out_dir=Path("out/tutorial_01_first_build_python/cifs"),
# )
# print("attempted:", attempted)
# for summary in summaries:
#     print(summary.topology_id, summary.status, summary.score, summary.cif_path)

## 查看构建输出

CIF 可能出现在 `valid`、`warning` 或 `needs_optimization` 目录中；这些目录表示粗略的几何分类，而不是不同的化学体系。Notebook 3 会搜索所有这些子目录，不会预设某一个类别。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
Write-Host "--- summary.json (first 100 lines) ---"
Get-Content -LiteralPath "out/tutorial_01_first_build/summary.json" | Select-Object -First 100
Write-Host "--- exported CIF files ---"
Get-ChildItem -LiteralPath "out/tutorial_01_first_build/cifs" -Recurse -File -Filter "*.cif" | Sort-Object FullName | ForEach-Object { $_.FullName }

In [ ]:
# Python 等价实现（已注释）：查看结构化报告并查找 CIF。
# import json
# from pathlib import Path
# output_dir = Path("out/tutorial_01_first_build")
# summary = json.loads((output_dir / "summary.json").read_text())
# print(summary["successful_structures"], summary["cifs_written"])
# print(*sorted((output_dir / "cifs").rglob("*.cif")), sep="\n")

## 需要保留和关注的内容

- `summary.json` 记录了所请求的化学体系、检测到的基元数量、尝试的结构、评分、验证标记和 CIF 路径。
- 每个 CIF 都以 `# COFid:` 注释开头，其中包含单体、拓扑和连接键。
- 这些是初始生成结构。标记为警告或 `needs_optimization` 的结果，应在模拟或发表前完成优化并重新验证。
- 请保留 `out/tutorial_01_first_build/`：Notebook 3 会把其中的 CIF 作为默认 Zeo++ 输入。

继续学习 [Notebook 2](02_topologies_connectivities_and_linkages.ipynb)，比较少量具有代表性的其他构建选择。